# LLMSTU dataset enhancement — Colab
Detect + crop students, pseudo-label with Qwen3-VL, then build a golden eval set.

**Runtime → Change runtime type → GPU** (T4 for 4B, A100 for 8B/30B). Then Run All.

## 1. Setup

In [ ]:
# Clone your repo. If you pushed this folder to GitHub, use that URL.
# Otherwise upload the LLMSTU-pipeline folder to the Colab file browser and skip this.
REPO_URL = ""  # e.g. "https://github.com/WaelK/LLMSTU-pipeline"
import os
if REPO_URL:
    !git clone $REPO_URL
    %cd LLMSTU-pipeline
elif os.path.isdir("LLMSTU-pipeline"):
    %cd LLMSTU-pipeline
print("cwd:", os.getcwd())

In [ ]:
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# HF token (needs write access to push results). Use Colab secrets or paste here.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ["HF_TOKEN"] = ""  # <-- paste hf_xxx if not using Colab secrets
assert os.environ["HF_TOKEN"], "Set HF_TOKEN"
from huggingface_hub import login; login(os.environ["HF_TOKEN"])

## 2. Config\nPick model size for your GPU. Start with a small `LIMIT` to sanity-check, then set it to `None`.

In [ ]:
from llmstu.config import load
cfg = load("config.yaml")
cfg.caption.model_id = "Qwen/Qwen3-VL-4B-Instruct"   # 8B/30B-A3B if you have A100
cfg.crop.detector_model = "yolo11m.pt"
LIMIT = 300   # crops to caption; set None for the full run
cfg.to_dict()

## 3. Detect + crop students\nDownloads frames from `WaelK/LLMSTU` (first run only), then crops.

In [ ]:
from pathlib import Path
from llmstu import dataset_io as io, crop as crop_mod
frames_dir = io.download_frames(cfg.data.source_repo, cfg.data.source_subdir,
                                Path(cfg.data.workdir)/"frames_dl", os.environ["HF_TOKEN"])
crop_mod.run(io.iter_frames(frames_dir, cfg.data.frames_glob),
             Path("work/crops"), cfg.crop, manifest_path=Path("work/crops_manifest.jsonl"))

In [ ]:
# peek at a few crops
import glob
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob("work/crops/*.jpg"))[:6]:
    display(IPImage(p, width=200))

## 4. Pseudo-label with Qwen3-VL

In [ ]:
from llmstu import caption as caption_mod
caption_mod.run(Path("work/crops_manifest.jsonl"), Path("work/crops"),
                Path("work/pseudo_labels.jsonl"), cfg.caption, limit=LIMIT)

In [ ]:
import json, pandas as pd
rows=[json.loads(l) for l in open("work/pseudo_labels.jsonl")]
df=pd.DataFrame(rows); print(len(df),"labeled")
df[["crop_path","activity","gaze_direction","engagement_level","model_confidence"]].head(10)

## 5. Build the golden-eval sample\nCreates a self-contained `labeling/data.json` (images embedded). Download the whole `labeling/` folder and open `index.html` to hand-correct.

In [ ]:
!python scripts/03_sample_eval.py --labels work/pseudo_labels.jsonl --crops work/crops --n 500 --embed
from google.colab import files
!cd labeling && zip -qr /content/labeling.zip index.html data.json
files.download("/content/labeling.zip")  # unzip locally, open index.html

## 6. Package + push\nAfter you've hand-labeled and have `golden.json`, upload it back to `labeling/` in Colab, then run this.

In [ ]:
!python scripts/04_upload.py --crops work/crops --labels work/pseudo_labels.jsonl \
    --golden labeling/golden.json --repo WaelK/LLMSTU-crops